# 🌫️ Air Quality Index (AQI) Analysis – India (April 2022 – April 2025)
**Submitted by:** Akriti Singh  
**Dataset:** aqi.csv — Daily city-level AQI readings across Indian states  
**Objective:** Clean the data, explore AQI trends, visualise pollutant patterns, and build a simple AQI category prediction model.

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')

---
## 2. Load & Inspect the Dataset

In [ ]:
df = pd.read_csv('aqi.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
df.describe()

---
## 3. Data Cleaning

In [ ]:
# ── 3.1  Drop redundant columns (unit, note carry no analytical value) ──
df.drop(columns=['unit', 'note'], inplace=True)

# ── 3.2  Parse dates ──
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

# ── 3.3  Standardise column names ──
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# ── 3.4  Strip whitespace from string columns ──
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# ── 3.5  Drop rows with missing aqi_value or air_quality_status ──
before = len(df)
df.dropna(subset=['aqi_value', 'air_quality_status'], inplace=True)
print(f'Rows dropped due to missing AQI / status: {before - len(df)}')

# ── 3.6  Remove duplicates ──
dup_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Duplicate rows removed: {dup_count}')

# ── 3.7  Remove non-positive AQI values (physically invalid) ──
invalid = (df['aqi_value'] <= 0).sum()
df = df[df['aqi_value'] > 0]
print(f'Invalid AQI rows removed: {invalid}')

# ── 3.8  Add helper time columns ──
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

print(f'\nClean dataset shape: {df.shape}')
df.head()

In [ ]:
# AQI category distribution after cleaning
print('AQI Category counts:')
print(df['air_quality_status'].value_counts())

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1  Overall AQI statistics ──
print('Overall AQI Statistics')
print('=' * 40)
print(df['aqi_value'].describe().rename({
    'count': 'Count', 'mean': 'Mean', 'std': 'Std Dev',
    'min': 'Min', '25%': '25th Pct', '50%': 'Median',
    '75%': '75th Pct', 'max': 'Max'
}))

In [ ]:
# ── 4.2  States with highest average AQI ──
state_avg = (df.groupby('state')['aqi_value']
               .mean()
               .sort_values(ascending=False)
               .reset_index())
state_avg.columns = ['State', 'Avg AQI']
print('Top 10 States by Average AQI:')
print(state_avg.head(10).to_string(index=False))

In [ ]:
# ── 4.3  Top 10 most polluted cities ──
city_avg = (df.groupby('area')['aqi_value']
              .mean()
              .sort_values(ascending=False)
              .reset_index())
city_avg.columns = ['City', 'Avg AQI']
print('Top 10 Most Polluted Cities (by Avg AQI):')
print(city_avg.head(10).to_string(index=False))

In [ ]:
# ── 4.4  Prominent pollutant frequency ──
# Expand comma-separated pollutants into individual entries
pollutant_series = (df['prominent_pollutants']
                    .dropna()
                    .str.split(',')
                    .explode()
                    .str.strip())
pollutant_counts = pollutant_series.value_counts()
print('Pollutant occurrence frequency:')
print(pollutant_counts)

In [ ]:
# ── 4.5  Monthly average AQI trend (across all years) ──
monthly_avg = (df.groupby(['year', 'month'])['aqi_value']
                 .mean()
                 .reset_index())
monthly_avg['period'] = pd.to_datetime(
    monthly_avg['year'].astype(str) + '-' + monthly_avg['month'].astype(str),
    format='%Y-%m'
)
monthly_avg.sort_values('period', inplace=True)
print(f'Date range in dataset: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Unique states: {df["state"].nunique()}')
print(f'Unique cities: {df["area"].nunique()}')

---
## 5. Visualisations

### Chart 1 — AQI Category Distribution (Pie Chart)

In [ ]:
status_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
status_colors = {
    'Good': '#2ecc71',
    'Satisfactory': '#a8d5a2',
    'Moderate': '#f39c12',
    'Poor': '#e67e22',
    'Very Poor': '#e74c3c',
    'Severe': '#922b21'
}

cat_counts = df['air_quality_status'].value_counts()
# Keep only categories present in data, in preferred order
present_cats = [c for c in status_order if c in cat_counts.index]
sizes  = cat_counts[present_cats].values
colors = [status_colors[c] for c in present_cats]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    sizes, labels=present_cats, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.2)
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title('Distribution of AQI Categories\n(All Cities, April 2022 – April 2025)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart1_aqi_distribution.png', bbox_inches='tight')
plt.show()
print('Chart 1 saved.')

### Chart 2 — Monthly Average AQI Trend (Line Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(monthly_avg['period'], monthly_avg['aqi_value'],
        color='steelblue', linewidth=2, marker='o', markersize=3, label='Avg AQI')

# Reference lines
ax.axhline(100, color='orange', linestyle='--', linewidth=1, label='Moderate threshold (100)')
ax.axhline(200, color='red',    linestyle='--', linewidth=1, label='Poor threshold (200)')

ax.set_title('Monthly Average AQI Trend (India, Apr 2022 – Apr 2025)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Average AQI', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart2_monthly_trend.png', bbox_inches='tight')
plt.show()
print('Chart 2 saved.')

### Chart 3 — Top 10 Most Polluted States (Bar Chart)

In [ ]:
top10_states = state_avg.head(10).copy()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_states['State'][::-1],
               top10_states['Avg AQI'][::-1],
               color='tomato', edgecolor='white')

for bar in bars:
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.0f}', va='center', fontsize=9)

ax.set_title('Top 10 States by Average AQI (Apr 2022 – Apr 2025)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Average AQI', fontsize=11)
ax.set_ylabel('State', fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('chart3_top_states.png', bbox_inches='tight')
plt.show()
print('Chart 3 saved.')

### Chart 4 — Prominent Pollutant Frequency (Bar Chart)

In [ ]:
poll_colors = {
    'PM10':  '#e74c3c',
    'PM2.5': '#c0392b',
    'O3':    '#3498db',
    'CO':    '#8e44ad',
    'NO2':   '#f39c12',
    'SO2':   '#27ae60',
    'NH3':   '#1abc9c',
    'Pb':    '#95a5a6'
}
poll_plot = pollutant_counts.head(8)
bar_colors = [poll_colors.get(p, '#7f8c8d') for p in poll_plot.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(poll_plot.index, poll_plot.values,
              color=bar_colors, edgecolor='white', width=0.6)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{bar.get_height():,.0f}', ha='center', fontsize=9)

ax.set_title('Prominent Pollutant Occurrence Frequency\n(All Records, Apr 2022 – Apr 2025)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pollutant', fontsize=11)
ax.set_ylabel('Number of Occurrences', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('chart4_pollutant_frequency.png', bbox_inches='tight')
plt.show()
print('Chart 4 saved.')

### Chart 5 — Seasonal AQI Pattern (Average AQI by Month Number)

In [ ]:
month_order = list(range(1, 13))
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

seasonal = (df.groupby('month')['aqi_value']
              .mean()
              .reindex(month_order))

# Colour bars by AQI level
def aqi_color(val):
    if val <= 50:   return '#2ecc71'
    if val <= 100:  return '#a8d5a2'
    if val <= 200:  return '#f39c12'
    if val <= 300:  return '#e67e22'
    return '#e74c3c'

bar_clrs = [aqi_color(v) for v in seasonal.values]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(month_labels, seasonal.values, color=bar_clrs, edgecolor='white', width=0.65)

for i, v in enumerate(seasonal.values):
    ax.text(i, v + 1.5, f'{v:.0f}', ha='center', fontsize=9)

ax.set_title('Seasonal AQI Pattern — Average AQI by Month\n(Across All Years in Dataset)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Average AQI', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71',  label='Good (≤50)'),
    Patch(facecolor='#a8d5a2',  label='Satisfactory (51–100)'),
    Patch(facecolor='#f39c12',  label='Moderate (101–200)'),
    Patch(facecolor='#e67e22',  label='Poor (201–300)'),
    Patch(facecolor='#e74c3c',  label='Very Poor (>300)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('chart5_seasonal_pattern.png', bbox_inches='tight')
plt.show()
print('Chart 5 saved.')

---
## 6. Key Insights

In [ ]:
overall_mean = df['aqi_value'].mean()
overall_median = df['aqi_value'].median()
worst_state = state_avg.iloc[0]
best_state  = state_avg.iloc[-1]
worst_city  = city_avg.iloc[0]
most_common_pollutant = pollutant_counts.index[0]
worst_month_idx = seasonal.idxmax()
best_month_idx  = seasonal.idxmin()

print('═' * 55)
print('  KEY INSIGHTS — India AQI Analysis (Apr 2022 – Apr 2025)')
print('═' * 55)
print(f'  Overall mean AQI    : {overall_mean:.1f}')
print(f'  Overall median AQI  : {overall_median:.1f}')
print(f'  Most polluted state : {worst_state["State"]} (avg AQI {worst_state["Avg AQI"]:.1f})')
print(f'  Cleanest state      : {best_state["State"]} (avg AQI {best_state["Avg AQI"]:.1f})')
print(f'  Most polluted city  : {worst_city["City"]} (avg AQI {worst_city["Avg AQI"]:.1f})')
print(f'  Dominant pollutant  : {most_common_pollutant} ({pollutant_counts.iloc[0]:,} occurrences)')
print(f'  Worst month (avg)   : {month_labels[worst_month_idx-1]} (AQI {seasonal[worst_month_idx]:.1f})')
print(f'  Best month (avg)    : {month_labels[best_month_idx-1]} (AQI {seasonal[best_month_idx]:.1f})')

# % of records in each category
pct = df['air_quality_status'].value_counts(normalize=True) * 100
print()
print('  AQI Category breakdown:')
for cat in present_cats:
    if cat in pct.index:
        print(f'    {cat:<15}: {pct[cat]:.1f}%')
print('═' * 55)

---
## 7. Simple AQI Category Prediction Model

Since the dataset lacks numeric sub-pollutant readings, we use **AQI value** and **month** as features to predict the **AQI category** (Good / Satisfactory / Moderate / Poor / Very Poor / Severe) using a basic **Decision Tree Classifier**. This serves as a simple, interpretable baseline.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# ── Feature engineering ──
model_df = df[['aqi_value', 'month', 'number_of_monitoring_stations',
               'air_quality_status']].dropna().copy()

le = LabelEncoder()
model_df['label'] = le.fit_transform(model_df['air_quality_status'])

X = model_df[['aqi_value', 'month', 'number_of_monitoring_stations']]
y = model_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=6, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Decision Tree — Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print()
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0
))

In [ ]:
# ── Feature importance ──
importances = pd.Series(clf.feature_importances_,
                        index=['AQI Value', 'Month', 'No. Stations'])

fig, ax = plt.subplots(figsize=(7, 4))
importances.sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Decision Tree — Feature Importances', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('chart6_feature_importance.png', bbox_inches='tight')
plt.show()
print('Feature importance chart saved.')

---
## 8. Summary

| Finding | Value |
|---|---|
| Total records analysed | See shape above |
| Date range | April 2022 – April 2025 |
| Most common AQI status | Satisfactory / Moderate |
| Dominant pollutant | PM10 |
| Winter months (Nov–Jan) | Highest average AQI |
| Monsoon months (Jul–Sep) | Lowest average AQI |
| Model (Decision Tree) | High accuracy — AQI value is the dominant feature |

> **Note:** The high model accuracy is expected because the AQI category labels are derived directly from the AQI value using fixed index thresholds (Good ≤ 50, Satisfactory 51–100, Moderate 101–200, etc.). The model confirms the data is internally consistent.